# 🌕 LUNAR HERITAGE – Fine-tune AI model
## Biến Llama 3.2 thành Thầy Đồ Neon chuyên di sản Việt Nam

**Cần làm trước:**
1. Menu **Runtime → Change runtime type → T4 GPU**
2. Chạy từng ô theo thứ tự
3. Toàn bộ quá trình mất khoảng **3–5 giờ**

In [ ]:
# ── Ô 1: Cài thư viện ──────────────────────────────────
!pip install -q unsloth transformers datasets trl peft bitsandbytes
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
print('✅ Cài xong thư viện!')

In [ ]:
# ── Ô 2: Kiểm tra GPU ──────────────────────────────────
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ Không có GPU!')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('CUDA:', torch.version.cuda)

In [ ]:
# ── Ô 3: Tải model Llama 3.2 (3B) ──────────────────────
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct'  # 3B = vừa đủ, chạy tốt trên T4
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name  = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,   # Tiết kiệm VRAM, chạy được trên T4 miễn phí
    dtype          = None,
)
print('✅ Đã tải model:', MODEL_NAME)

In [ ]:
# ── Ô 4: Cấu hình LoRA (kỹ thuật fine-tune hiệu quả) ───
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,      # rank – càng cao càng học sâu, tốn VRAM hơn
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                       'gate_proj','up_proj','down_proj'],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
print('✅ LoRA cấu hình xong!')
print('Tham số cần train:', f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# ── Ô 5: Upload và xử lý dataset ───────────────────────
# Upload file heritage_dataset.jsonl từ máy tính lên Colab
from google.colab import files
print('Chọn file heritage_dataset.jsonl để upload:')
uploaded = files.upload()

import json
from datasets import Dataset

# Đọc dataset
records = []
with open('heritage_dataset.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'✅ Đã đọc {len(records)} mẫu training')

# Format theo ChatML
def format_chat(sample):
    messages = sample['messages']
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': text}

raw_dataset = Dataset.from_list(records)
dataset = raw_dataset.map(format_chat)
print('✅ Dataset sẵn sàng!')
print('Ví dụ mẫu đầu tiên:')
print(dataset[0]['text'][:300], '...')

In [ ]:
# ── Ô 6: Cấu hình Training ────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = dataset,
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ_LEN,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 4,
        warmup_steps       = 5,
        num_train_epochs   = 3,     # Học 3 lần qua toàn bộ dataset
        learning_rate      = 2e-4,
        fp16               = not torch.cuda.is_bf16_supported(),
        bf16               = torch.cuda.is_bf16_supported(),
        logging_steps      = 10,
        optim              = 'adamw_8bit',
        weight_decay       = 0.01,
        lr_scheduler_type  = 'linear',
        seed               = 42,
        output_dir         = './lunar_heritage_model',
        save_strategy      = 'epoch',
    ),
)
print('✅ Trainer sẵn sàng!')

In [ ]:
# ── Ô 7: BẮT ĐẦU TRAIN ────────────────────────────────
# ⏱ Thời gian: khoảng 2–4 giờ với T4 GPU
print('🚀 Bắt đầu fine-tune Thầy Đồ Neon...')
print('Ước tính: 2–4 giờ với GPU T4 Colab')

trainer_stats = trainer.train()

print('\n✅ Training xong!')
print(f'Tổng thời gian: {trainer_stats.metrics["train_runtime"] / 60:.1f} phút')
print(f'Loss cuối: {trainer_stats.metrics["train_loss"]:.4f}')

In [ ]:
# ── Ô 8: Test model trước khi lưu ─────────────────────
FastLanguageModel.for_inference(model)

def chat_test(question):
    messages = [
        {"role": "system", "content": "Bạn là Thầy Đồ Neon – AI chuyên về di sản văn hóa Việt Nam."},
        {"role": "user",   "content": question}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=300,
        temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'\n❓ Câu hỏi: {question}')
    print(f'💬 Trả lời: {response}')
    return response

# Test 3 câu hỏi mẫu
chat_test('Vịnh Hạ Long được UNESCO công nhận năm nào?')
chat_test('Món ăn đặc trưng của Huế là gì?')
chat_test('Hội An nổi tiếng về điều gì?')

In [ ]:
# ── Ô 9: Lưu model ra file .gguf (dùng được với Ollama) ─
print('💾 Đang lưu model...')

# Lưu LoRA weights
model.save_pretrained('./lunar_heritage_lora')
tokenizer.save_pretrained('./lunar_heritage_lora')
print('✅ Lưu LoRA weights xong!')

# Export sang GGUF (dùng được với Ollama offline)
model.save_pretrained_gguf(
    './lunar_heritage_gguf',
    tokenizer,
    quantization_method = 'q4_k_m'  # Nén 4-bit, chạy tốt trên CPU thường
)
print('✅ Export GGUF xong!')

import os
gguf_files = [f for f in os.listdir('./lunar_heritage_gguf') if f.endswith('.gguf')]
print('File GGUF:', gguf_files)

In [ ]:
# ── Ô 10: Tải model về máy ────────────────────────────
# Nén và tải về máy tính của bạn
import shutil
shutil.make_archive('lunar_heritage_model', 'zip', './lunar_heritage_gguf')

from google.colab import files
files.download('lunar_heritage_model.zip')
print('✅ Bắt đầu tải file về máy!')
print('File GGUF này sẽ dùng với Ollama ở bước tiếp theo.')